# MiniLM S2 + leakage-safe CatBoost on three external splits

The transformer is frozen. CatBoost is fit on old leakage-safe S2
holdout predictions after removing every product present in IID,
hard, or OOD. Evaluation is external and item-disjoint.

In [ ]:
import hashlib
import json
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path, PurePosixPath

import pandas as pd

INPUT_ROOT = Path('/kaggle/input')
WORKING_ROOT = Path('/kaggle/working')
PROJECT_ROOT = Path('/kaggle/temp/s2_catboost_new_splits/project')
OUTPUT_DIR = WORKING_ROOT / 's2_catboost_new_splits'
EXPECTED_BUNDLE_SHA256 = '7d39de84c3a8da704a3f71708c9e6f8309a38efcc2e743472e685a3173d361f3'
EXPECTED_SOURCE_MANIFEST = {'schema_version': 1, 'files': {'configs/cheap_ensemble_s2.json': {'bytes': 587, 'sha256': '3b02955f4d2c53d76dda1c211a6209cc822dd5a8796a5a3fa647318331901ef1'}, 'src/__init__.py': {'bytes': 62, 'sha256': '1ebfb0504084e6c7b27bb3a60370eb9f01bdf6ac9a801bc400a58da4d8d674eb'}, 'src/cheap_ensemble.py': {'bytes': 13915, 'sha256': 'b9fee08e2a07d2424fd33f39c87bc250d73d1966ee6354a16e1ad4848f253afc'}, 'scripts/train_s2_cheap_ensemble.py': {'bytes': 14913, 'sha256': 'aa396314fdfcacd7bc64ed9217322d93080dc796987bf122e02958525e67155c'}, 'scripts/evaluate_s2_catboost_new_splits.py': {'bytes': 10199, 'sha256': '4b0adf8cbff0268e38eee6bcc31bcd705d6a7847cc28b639ed1a06e440fe2ebb'}, 'artifacts/manual/S2_VALUES_ONLY/validation_predictions.parquet': {'bytes': 1738244, 'sha256': '91c7ab66fefefdd696e32028d0874f96f0b2ae446d486c94192c3fef43c9c082'}, 'artifacts/kaggle/product-matching-minilm-s0-s2-new-splits/minilm_s0_s2_new_splits/evaluations/S2_VALUES_ONLY/iid/predictions.parquet': {'bytes': 427409, 'sha256': '439dc7ddfc4bce2de52151ce649d9513be72ad3292323d314c64ae1073deafc6'}, 'artifacts/kaggle/product-matching-minilm-s0-s2-new-splits/minilm_s0_s2_new_splits/evaluations/S2_VALUES_ONLY/hard/predictions.parquet': {'bytes': 216602, 'sha256': '233c531d4da5deeb54c7a0d36897cf386e7df4a35fdd25bcfe12c2a240ddb4b9'}, 'artifacts/kaggle/product-matching-minilm-s0-s2-new-splits/minilm_s0_s2_new_splits/evaluations/S2_VALUES_ONLY/ood/predictions.parquet': {'bytes': 1358943, 'sha256': '6f7a2d751064962f84dc8a23e359f3eaefbaad6354348cdaaf92216ea5bd2796'}}}

def exactly_one(filename):
    candidates = list(INPUT_ROOT.glob(f'**/{filename}'))
    if len(candidates) != 1:
        raise RuntimeError(f'Expected exactly one {filename!r}, found {candidates}')
    return candidates[0]

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

items_path = exactly_one('items_human.parquet')
split_manifest = exactly_one('validation_splits_manifest.json')
bundle_candidates = list(INPUT_ROOT.glob('**/s2_catboost_new_splits_bundle.zip'))
bundle_candidates.extend(
    path for path in INPUT_ROOT.glob('**/s2_catboost_new_splits_bundle') if path.is_dir()
)
if len(bundle_candidates) != 1:
    raise RuntimeError(f'Expected one bundle, found {bundle_candidates}')
bundle_path = bundle_candidates[0]
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
if bundle_path.is_file():
    if sha256(bundle_path) != EXPECTED_BUNDLE_SHA256:
        raise RuntimeError('Bundle SHA-256 mismatch')
    with zipfile.ZipFile(bundle_path) as archive:
        for member in archive.namelist():
            relative = PurePosixPath(member)
            if relative.is_absolute() or '..' in relative.parts:
                raise RuntimeError(f'Unsafe bundle member: {member}')
        archive.extractall(PROJECT_ROOT)
else:
    shutil.copytree(bundle_path, PROJECT_ROOT, dirs_exist_ok=True)
source_manifest = json.loads(
    (PROJECT_ROOT / 'source_manifest.json').read_text(encoding='utf-8')
)
if source_manifest != EXPECTED_SOURCE_MANIFEST:
    raise RuntimeError('Source manifest mismatch')
for relative, expected in source_manifest['files'].items():
    source = PROJECT_ROOT.joinpath(*PurePosixPath(relative).parts)
    if sha256(source) != expected['sha256']:
        raise RuntimeError(f'Source hash mismatch: {relative}')
split_dir = split_manifest.parent
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('items:', items_path)
print('split_dir:', split_dir)
print('bundle:', bundle_path)

## Dependencies and evaluation

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet',
     '--disable-pip-version-check', 'catboost==1.2.8',
     'rapidfuzz==3.14.5'],
    check=True,
)
command = [
    sys.executable, '-u',
    str(PROJECT_ROOT / 'scripts/evaluate_s2_catboost_new_splits.py'),
    '--items', str(items_path),
    '--meta-predictions', str(
        PROJECT_ROOT / 'artifacts/manual/S2_VALUES_ONLY/validation_predictions.parquet'
    ),
    '--split-dir', str(split_dir),
    '--s2-evaluations-dir', str(
        PROJECT_ROOT / 'artifacts/kaggle/product-matching-minilm-s0-s2-new-splits/'
        'minilm_s0_s2_new_splits/evaluations/S2_VALUES_ONLY'
    ),
    '--config', str(PROJECT_ROOT / 'configs/cheap_ensemble_s2.json'),
    '--output-dir', str(OUTPUT_DIR),
]
subprocess.run(command, check=True, cwd=PROJECT_ROOT)

## Result

In [ ]:
report = json.loads((OUTPUT_DIR / 'evaluation_report.json').read_text(encoding='utf-8'))
rows = []
for split, values in report['split_reports'].items():
    rows.append({
        'split': split,
        'S2_macro_AP': values['transformer']['macro_average_precision'],
        'S2_CatBoost_macro_AP': values['catboost']['macro_average_precision'],
        'delta': values['absolute_delta_macro_ap'],
    })
display(pd.DataFrame(rows))
display(pd.DataFrame(report['top_catboost_features']))
if not (OUTPUT_DIR / 'COMPLETED').is_file():
    raise RuntimeError('Completion marker missing')